## Import

In [1]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import DataLoader, random_split
from PIL import Image
from pathlib import Path
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchmetrics.detection.mean_ap import MeanAveragePrecision

import comet_ml

In [2]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, image_dir, label_dir):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.images = sorted(self.image_dir.glob("*.*"))

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label_path = self.label_dir / (img_path.stem + ".txt")

        img = Image.open(img_path).convert("RGB")
        img_w, img_h = img.size

        img = torchvision.transforms.functional.to_tensor(img)

        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path) as f:
                for line in f:
                    cls, x, y, w, h = map(float, line.split())

                    xmin = (x - w / 2) * img_w
                    ymin = (y - h / 2) * img_h
                    xmax = (x + w / 2) * img_w
                    ymax = (y + h / 2) * img_h

                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(int(cls) + 1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if len(boxes) > 0:
            boxes[:, 0::2] = boxes[:, 0::2].clamp(min=0, max=img_w)
            boxes[:, 1::2] = boxes[:, 1::2].clamp(min=0, max=img_h)

            widths = boxes[:, 2] - boxes[:, 0]
            heights = boxes[:, 3] - boxes[:, 1]

            valid = (
                (boxes[:, 2] > boxes[:, 0]) &
                (boxes[:, 3] > boxes[:, 1]) &
                (widths >= 4) &
                (heights >= 4)
            )

            boxes = boxes[valid]
            labels = labels[valid]
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
        }

        return img, target

    def __len__(self):
        return len(self.images)

In [3]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    return tuple(zip(*batch))

dataset = Dataset("images", "labels")

train_size = int(0.7*len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=collate_fn
)

## Model

In [4]:
num_classes = 2

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(device)

cuda


In [5]:
lr = 1e-3
optimizer = torch.optim.SGD(model.parameters(), lr = lr, momentum=0.9, weight_decay=0.0005)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.3, patience=5, verbose=True) # Verbose is deprecated --> current_lrs = scheduler.get_last_lr()
Early_Stop_Patience = 12
patience_counter = 0
epochs = 200

c:\Users\david\anaconda3\envs\tretolv\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


### Logging Setup

In [9]:
# initialize comet
project_ml = comet_ml.Experiment(
    api_key="wCXnRD5xewUGxCxBYe8ePt4JY",
    workspace="kanskejoanna",
    project_name="car-detection-in-snow",
    name="Faster R-CNN Training",
    display_name="Faster R-CNN Training",
)

# Report multiple hyperparameters using a dictionary:
hyper_params = {
   "learning_rate": lr,
   "steps": epochs
}
project_ml.log_parameters(hyper_params)

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: sklearn, torch.


COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : respective_balustrade_4590
COMET INFO:     url                   : https://www.comet.com/kanskejoanna/car-detection-in-snow/af74d97c002e49f1b48a45f8ddf086c9
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     Train Loss [4] : (0.07182551891872932, 0.16013206489714135)
COMET INFO:     Val Loss [4]   : (0.0957171959458771, 0.11981712948466278)
COMET INFO:   Parameters:
COMET INFO:     learning_rate : 0.001
COMET INFO:     steps         : 200
COMET INFO:   Uploads:
COMET INFO:     environment details      : 1
COMET INFO:     filename                 : 1
COMET INFO:     git metadata             : 1
COMET INFO:     git-patch (uncompressed) : 1 (12.2

# Training and Validaiton

In [ ]:
best_val_loss = float("inf")
best_model_path = "Faster_RCNN.pth"

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for imgs, targets in train_loader:

        imgs = [img.to(device) for img in imgs]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, targets) # Loss = loss_classifier, loss_box_reg, loss_objectness and loss_rpn_box_reg.
        loss = sum(loss_dict.values()) # Total loss for backpropagation

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    model.train()
    val_loss = 0

    with torch.no_grad():
        for imgs, targets in val_loader:
            imgs = [img.to(device) for img in imgs]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

            loss_dict = model(imgs, targets) 
            loss = sum(loss_dict.values())

            val_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)

    scheduler.step(avg_val_loss)

    project_ml.log_metric("Train Loss", avg_train_loss)
    project_ml.log_metric("Val Loss", avg_val_loss)

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model with Val Loss: {best_val_loss:.3f}")
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= Early_Stop_Patience:
            print(f"Early stopping initiated after {epoch+1} epochs")
            break

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.3f} | "
        f"Val Loss: {avg_val_loss:.3f}"
        )
    
project_ml.end()

AttributeError: 'Experiment' object has no attribute 'start'

In [ ]:
model.load_state_dict(torch.load("Faster_RCNN.pth", map_location=device))
model.to(device)
model.eval()

metric = MeanAveragePrecision()

with torch.no_grad():
    for imgs, targets in test_loader:
        imgs = [img.to(device) for img in imgs]

        preds = model(imgs)

        preds = [
            {k: v.detach().cpu() for k, v in p.items()}
            for p in preds
        ]

        targets = [
            {k: v.detach().cpu() for k, v in t.items()}
            for t in targets
        ]

        metric.update(preds, targets)

results = metric.compute()

print(f"mAP: {results['map'].item() * 100:.2f}%")
print(f"mAP@50: {results['map_50'].item() * 100:.2f}%")



C:\Users\david\AppData\Local\Temp\ipykernel_68404\1281351254.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("Faster_RCNN.pth", map_loca

mAP: 83.74%
mAP@50: 97.96%


In [ ]:
with_objects = 0
without_objects = 0

for _, target in dataset:
    if len(target["boxes"]) > 0:
        with_objects += 1
    else:
        without_objects += 1

print("With objects:", with_objects)
print("Without objects:", without_objects)

With objects: 8450
Without objects: 0
